# inplace-param-update — ex1: in-place vs out-of-place parameter update

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inplace-param-update`. Running the final beacon cell reports progress against the `PyTorch: In-place param update` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: In-place param update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-param-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-param-update"
DD_SUBTOPIC = "PyTorch: In-place param update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## In-place parameter update — quick refresher

When you hand-roll an optimizer, the parameter update MUST happen in-place. Two valid forms:

```
theta -= lr * g                # in-place op via -=
theta.sub_(lr * g)             # in-place op via *_ method
```

**Why in-place.** A new-tensor update (`theta = theta - lr * g`) rebinds the LOCAL name to a brand-new tensor; the original `nn.Parameter` is untouched and the model still sees the old weights. The model holds a reference to the original storage; you have to mutate that storage.

**Inside `@t.inference_mode()` / `@t.no_grad()`.** Optimizer `.step()` methods are decorated with `@t.no_grad()` (or `@t.inference_mode()`) so the in-place mutation doesn't get tracked by autograd. Without that, the next `.backward()` would error: 'a leaf Variable that requires grad is being used in an in-place operation.'

### Exercise 1 — in-place vs out-of-place parameter update

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze why `theta -= lr * g` (in-place) correctly updates a model's parameters but `theta = theta - lr * g` (out-of-place) does not, by comparing the post-update tensor identities.
> Keywords: in-place, rebind, data-attribute
> ```

**KCs targeted:** `inplace-update-mutates-storage`, `out-of-place-update-rebinds-local-name`

Implement two functions that BOTH look like an SGD parameter update — but only ONE actually changes the model's weights. The test confirms the failure mode of the wrong one and the success of the right one.

**Implement `ex1_apply_update_inplace(param, grad, lr)`:**
- Mutate `param` IN PLACE: `param.data -= lr * grad`.
- Return nothing.
- After the call, the same `param` tensor still exists in its containing module — only its underlying storage is updated.

**Implement `ex1_apply_update_wrong(param, grad, lr)`:**
- Do the out-of-place version: `param = param - lr * grad` (simple Python rebind of the LOCAL name).
- This is the bug we're isolating. Return the rebound local name so the test can compare identities.

Inputs:
- `param`: an `nn.Parameter` (or leaf tensor) wrapped by a module — the test passes a real `nn.Linear`'s `.weight`.
- `grad`: a tensor with the same shape as `param`.
- `lr`: float.

The test then constructs a 1-layer `nn.Linear`, calls each function on its weight, and checks (a) `model.weight is` the same Python object before and after either call, (b) only the in-place version actually changed the underlying values.

In [ ]:
def ex1_apply_update_inplace(param, grad, lr):
    # `param.data` lets us mutate the storage outside the autograd graph.
    # Equivalent inside a no_grad/inference_mode block: `param -= lr * grad`.
    param.data -= lr * grad


def ex1_apply_update_wrong(param, grad, lr):
    # This rebinds the LOCAL `param` name to a new tensor.
    # The model still holds a reference to the original — unchanged.
    param = param - lr * grad
    return param


<details><summary>Solution</summary>

```python
def ex1_apply_update_inplace(param, grad, lr):
    # `param.data` lets us mutate the storage outside the autograd graph.
    # Equivalent inside a no_grad/inference_mode block: `param -= lr * grad`.
    param.data -= lr * grad


def ex1_apply_update_wrong(param, grad, lr):
    # This rebinds the LOCAL `param` name to a new tensor.
    # The model still holds a reference to the original — unchanged.
    param = param - lr * grad
    return param
```

**Why `.data` instead of `param -= lr * grad`.** Outside a `no_grad` context, the bare in-place op on a leaf with `requires_grad=True` raises: 'a leaf Variable that requires grad is being used in an in-place operation.' Real PyTorch optimizers wrap `.step()` in `@t.no_grad()` (or `@t.inference_mode()`) so they can write `param -= lr * grad` without the `.data` workaround. Using `.data` is the battle-tested escape hatch when you want to mutate weights from outside autograd's view.

**The Python aliasing model is the real lesson.** In Python, `x = x + 1` rebinds the local name `x` — it does not mutate the original object. `x += 1` calls `__iadd__` which mutates in place for mutable types (lists, tensors). Tensor parameters live inside `nn.Module` containers via attribute reference; you have to mutate the storage to be seen.

**ARENA SGD impl gotcha.** The solution comment literally reads `theta -= self.lr * g  # inplace operation, to modify params`. This is THE most-explained line in the chap-0 part-3 optimizer exercise. The drill here isolates the lesson.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()